# Ablation Study: Losses, Augmentations, Thresholds

Runs executable ablations on Kvasir-SEG outputs: checkpoint threshold sweeps and optional train-time ablations.


In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)


BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:
import json
import os
from pathlib import Path

import pandas as pd
import torch

from utils.segmentation_common import (
    TrainConfig,
    UNetSmall,
    build_deeplab_resnet50,
    evaluate_model,
    find_kvasir_seg_root,
    load_metadata,
    load_model_from_checkpoint,
    make_eval_dataloader,
    read_run_config,
    summarize_metric_frame,
    train_and_evaluate,
)

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
SPLIT_HASH_TXT = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'split_hash.txt'
OUT_DIR = ROOT / '3_generalization_and_ablation' / 'out' / 'ablations'
OUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLDS = [float(x.strip()) for x in os.getenv('ABLATION_THRESHOLDS', '0.3,0.4,0.5,0.6,0.7').split(',') if x.strip()]
MAX_EVAL_SAMPLES = int(os.getenv('MAX_EVAL_SAMPLES', '0')) or None
EVAL_BATCH_SIZE = int(os.getenv('EVAL_BATCH_SIZE', '0')) or None
EVAL_NUM_WORKERS = os.getenv('EVAL_NUM_WORKERS', '').strip()
EVAL_NUM_WORKERS = int(EVAL_NUM_WORKERS) if EVAL_NUM_WORKERS else None
EVAL_IMAGE_SIZE = int(os.getenv('EVAL_IMAGE_SIZE', '0')) or None
COMPUTE_HD95 = os.getenv('COMPUTE_HD95', '0') == '1'
ALLOW_HF_DOWNLOAD = os.getenv('ALLOW_HF_DOWNLOAD', '0') == '1'

RUN_FILTER = [x.strip() for x in os.getenv('EVAL_RUN_FILTER', '').split(',') if x.strip()]

RUN_TRAIN_ABLATIONS = os.getenv('RUN_TRAIN_ABLATIONS', '0') == '1'
TRAIN_ABL_MODEL = os.getenv('TRAIN_ABL_MODEL', 'unet_small').strip().lower()
TRAIN_ABL_EPOCHS = int(os.getenv('TRAIN_ABL_EPOCHS', '4'))
TRAIN_ABL_BATCH_SIZE = int(os.getenv('TRAIN_ABL_BATCH_SIZE', '8'))
TRAIN_ABL_NUM_WORKERS = int(os.getenv('TRAIN_ABL_NUM_WORKERS', '2'))
TRAIN_ABL_IMAGE_SIZE = int(os.getenv('TRAIN_ABL_IMAGE_SIZE', '352'))
TRAIN_ABL_LR = float(os.getenv('TRAIN_ABL_LR', '1e-3'))
TRAIN_ABL_WEIGHT_DECAY = float(os.getenv('TRAIN_ABL_WEIGHT_DECAY', '1e-4'))
TRAIN_MAX_TRAIN = int(os.getenv('TRAIN_MAX_TRAIN_SAMPLES', '0')) or None
TRAIN_MAX_VAL = int(os.getenv('TRAIN_MAX_VAL_SAMPLES', '0')) or None
TRAIN_MAX_TEST = int(os.getenv('TRAIN_MAX_TEST_SAMPLES', '0')) or None

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('THRESHOLDS:', THRESHOLDS)
print('MAX_EVAL_SAMPLES:', MAX_EVAL_SAMPLES)
print('RUN_FILTER:', RUN_FILTER if RUN_FILTER else '<all>')
print('RUN_TRAIN_ABLATIONS:', RUN_TRAIN_ABLATIONS)
print('TRAIN_ABL_MODEL:', TRAIN_ABL_MODEL)
print('CUDA available:', torch.cuda.is_available())


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
OUT_DIR: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations
THRESHOLDS: [0.3, 0.4, 0.5, 0.6, 0.7]
MAX_EVAL_SAMPLES: None
RUN_FILTER: <all>
RUN_TRAIN_ABLATIONS: False
TRAIN_ABL_MODEL: unet_small
CUDA available: True


In [3]:
meta_df = load_metadata(META_CSV)

test_df = meta_df[meta_df['split'] == 'test'].copy().reset_index(drop=True)
if MAX_EVAL_SAMPLES:
    test_df = test_df.sample(n=min(MAX_EVAL_SAMPLES, len(test_df)), random_state=42).reset_index(drop=True)

print('Evaluation test samples:', len(test_df))
display(test_df.head())


Evaluation test samples: 100


,img_id,image_path,mask_path,width,height,fg_pixels,total_pixels,mask_area_ratio,component_count,bbox_count,split,split_seed
0,cju0qoxqj9q6s0835b43399p4,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,1348,1070,390295,1442360,0.270595,1,1,test,42
1,cju0tl3uz8blh0993wxvn7ly3,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,529,1735,329038,0.005273,1,1,test,42
2,cju13cgqmnhwn0988yrainhcp,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,626,546,42478,341796,0.124279,1,1,test,42
3,cju171py4qiha0835u8sl59ds,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,616,530,119613,326480,0.366372,2,1,test,42
4,cju17v6ih0u7808783zcbg1jy,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...,622,528,115218,328416,0.350829,2,1,test,42


In [4]:
search_roots = [
    ROOT / '1_classic_seg_baselines' / 'out',
    ROOT / '2_modern_segmentation' / 'out',
]

run_dirs = []
for sr in search_roots:
    if not sr.exists():
        continue
    for ckpt in sorted(sr.rglob('best_model.pt')):
        run_dirs.append(ckpt.parent)

run_dirs = sorted({p.resolve() for p in run_dirs})
if RUN_FILTER:
    run_dirs = [p for p in run_dirs if p.name in RUN_FILTER]

run_df = pd.DataFrame({'run_name': [p.name for p in run_dirs], 'run_dir': [str(p) for p in run_dirs]})
run_csv = OUT_DIR / 'available_runs_summary.csv'
run_df.to_csv(run_csv, index=False)

print('Discovered runs:', len(run_dirs))
print('Saved:', run_csv)
display(run_df)


Discovered runs: 3
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/available_runs_summary.csv


,run_name,run_dir
0,deeplabv3plus_resnet50_baseline,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...
1,unet_resnet34_baseline,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...
2,segformer_b2_finetune,/mnt/hf/thesis/rag-vqa-medical/Prototyping_ref...


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rows = []
failures = []

for run_dir in run_dirs:
    run_cfg = read_run_config(run_dir)
    model_name = str(run_cfg.get('model', run_dir.name))
    ckpt = run_dir / 'best_model.pt'

    print(f'\nThreshold sweep: {run_dir.name} ({model_name})')
    try:
        model = load_model_from_checkpoint(
            checkpoint_path=ckpt,
            model_name_hint=model_name,
            allow_hf_download=ALLOW_HF_DOWNLOAD,
        ).to(device)

        image_size = int(EVAL_IMAGE_SIZE if EVAL_IMAGE_SIZE is not None else run_cfg.get('image_size', 352))
        batch_size = int(EVAL_BATCH_SIZE if EVAL_BATCH_SIZE is not None else run_cfg.get('batch_size', 4))
        num_workers = int(EVAL_NUM_WORKERS if EVAL_NUM_WORKERS is not None else run_cfg.get('num_workers', 2))

        loss_cfg = TrainConfig(
            loss_name=str(run_cfg.get('loss_name', 'bce_dice')),
            bce_weight=float(run_cfg.get('bce_weight', 0.5)),
            focal_gamma=float(run_cfg.get('focal_gamma', 2.0)),
            focal_alpha=float(run_cfg.get('focal_alpha', 0.25)),
        )

        loader = make_eval_dataloader(
            metadata_df=test_df,
            image_size=image_size,
            batch_size=batch_size,
            num_workers=num_workers,
            seed=int(run_cfg.get('seed', 42)),
            split_name='test',
        )

        for th in THRESHOLDS:
            loss, frame, _ = evaluate_model(
                model=model,
                loader=loader,
                device=device,
                threshold=float(th),
                compute_hd95=COMPUTE_HD95,
                collect_pred_masks=False,
                loss_cfg=loss_cfg,
            )
            summary = summarize_metric_frame(frame)
            summary['loss'] = float(loss)
            row = {
                'run_name': run_dir.name,
                'model_name': model_name,
                'threshold': float(th),
                'n_images': int(len(frame)),
            }
            row.update(summary)
            rows.append(row)
            print(f"  th={th:.2f} dice={summary.get('dice_mean')} iou={summary.get('iou_mean')}")

    except Exception as e:
        failures.append({'run_name': run_dir.name, 'run_dir': str(run_dir), 'error': str(e)})
        print('  FAILED:', e)

sweep_df = pd.DataFrame(rows)
if not sweep_df.empty:
    sweep_df = sweep_df.sort_values(['run_name', 'threshold']).reset_index(drop=True)

fail_df = pd.DataFrame(failures)

sweep_csv = OUT_DIR / 'threshold_sweep.csv'
fail_csv = OUT_DIR / 'threshold_sweep_failures.csv'
sweep_df.to_csv(sweep_csv, index=False)
fail_df.to_csv(fail_csv, index=False)

print('Saved:', sweep_csv)
print('Saved:', fail_csv)
display(sweep_df.head(20))
if not fail_df.empty:
    display(fail_df)



Threshold sweep: deeplabv3plus_resnet50_baseline (deeplabv3_resnet50)
  th=0.30 dice=0.7235746820774627 iou=0.6136103092753072
  th=0.40 dice=0.7241981926432121 iou=0.6159694463002898
  th=0.50 dice=0.7181982553214378 iou=0.61109260500399
  th=0.60 dice=0.7057003956050565 iou=0.5995433645340047
  th=0.70 dice=0.6859351597380392 iou=0.5794882521091896

Threshold sweep: unet_resnet34_baseline (unet_small)
  th=0.30 dice=0.5733296998762154 iou=0.438843381093399
  th=0.40 dice=0.5745445209903862 iou=0.4412730807663053
  th=0.50 dice=0.5706663906452512 iou=0.4387758063289046
  th=0.60 dice=0.5607688310816836 iou=0.4316368531702973
  th=0.70 dice=0.5309867930994902 iou=0.405311572314263

Threshold sweep: segformer_b2_finetune (segformer_binary)


2026-02-15 10:51:58.448802: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


  th=0.30 dice=0.44395102102485157 iou=0.31829319990296345
  th=0.40 dice=0.46139312748641886 iou=0.3336569928902374
  th=0.50 dice=0.4748864772301102 iou=0.3451914223773132
  th=0.60 dice=0.480953859514519 iou=0.35115746023549144
  th=0.70 dice=0.4782359546901736 iou=0.35251854162735674
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/threshold_sweep.csv
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/threshold_sweep_failures.csv


,run_name,model_name,threshold,n_images,n,dice_mean,dice_median,dice_std,iou_mean,iou_median,...,recall_mean,recall_median,recall_std,f1_mean,f1_median,f1_std,specificity_mean,specificity_median,specificity_std,loss
0,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.3,100,100,0.723575,0.806909,0.237637,0.613610,0.676332,...,0.795446,0.895263,0.243536,0.723575,0.806909,0.237637,0.962932,0.979576,0.045856,0.292433
1,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.4,100,100,0.724198,0.819882,0.242168,0.615969,0.694747,...,0.759156,0.862673,0.260259,0.724198,0.819882,0.242168,0.972641,0.986935,0.035388,0.292433
2,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.5,100,100,0.718198,0.818413,0.250094,0.611093,0.692641,...,0.724707,0.828515,0.272905,0.718198,0.818413,0.250094,0.979375,0.990148,0.028990,0.292433
3,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.6,100,100,0.705700,0.803734,0.261960,0.599543,0.671875,...,0.687066,0.797569,0.284045,0.705700,0.803734,0.261960,0.984678,0.993176,0.024316,0.292433
4,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.7,100,100,0.685935,0.782612,0.274780,0.579488,0.642862,...,0.644140,0.748122,0.293079,0.685935,0.782612,0.274780,0.988815,0.996293,0.020629,0.292433
5,segformer_b2_finetune,segformer_binary,0.3,100,100,0.443951,0.446274,0.240838,0.318293,0.287239,...,0.849824,0.942242,0.209789,0.443951,0.446274,0.240838,0.742589,0.722906,0.103130,0.502461
6,segformer_b2_finetune,segformer_binary,0.4,100,100,0.461393,0.483074,0.241631,0.333657,0.318460,...,0.809858,0.901195,0.234638,0.461393,0.483074,0.241631,0.791183,0.781146,0.094718,0.502461
7,segformer_b2_finetune,segformer_binary,0.5,100,100,0.474886,0.494683,0.240895,0.345191,0.328639,...,0.760785,0.845456,0.260486,0.474886,0.494683,0.240895,0.834868,0.826673,0.083883,0.502461
8,segformer_b2_finetune,segformer_binary,0.6,100,100,0.480954,0.525512,0.244464,0.351157,0.356406,...,0.692310,0.771861,0.286356,0.480954,0.525512,0.244464,0.877158,0.873199,0.072991,0.502461
9,segformer_b2_finetune,segformer_binary,0.7,100,100,0.478236,0.506865,0.259590,0.352519,0.339478,...,0.585255,0.650502,0.297903,0.478236,0.506865,0.259590,0.923433,0.932278,0.059983,0.502461


In [6]:
if sweep_df.empty:
    best_df = pd.DataFrame()
else:
    best_idx = sweep_df.groupby('run_name')['dice_mean'].idxmax()
    best_df = sweep_df.loc[best_idx].sort_values('dice_mean', ascending=False).reset_index(drop=True)

best_csv = OUT_DIR / 'threshold_best_per_run.csv'
best_df.to_csv(best_csv, index=False)
print('Saved:', best_csv)
display(best_df)


Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/threshold_best_per_run.csv


,run_name,model_name,threshold,n_images,n,dice_mean,dice_median,dice_std,iou_mean,iou_median,...,recall_mean,recall_median,recall_std,f1_mean,f1_median,f1_std,specificity_mean,specificity_median,specificity_std,loss
0,deeplabv3plus_resnet50_baseline,deeplabv3_resnet50,0.4,100,100,0.724198,0.819882,0.242168,0.615969,0.694747,...,0.759156,0.862673,0.260259,0.724198,0.819882,0.242168,0.972641,0.986935,0.035388,0.292433
1,unet_resnet34_baseline,unet_small,0.4,100,100,0.574545,0.615454,0.239486,0.441273,0.444535,...,0.759421,0.840915,0.252739,0.574545,0.615454,0.239486,0.896991,0.917916,0.092909,0.399405
2,segformer_b2_finetune,segformer_binary,0.6,100,100,0.480954,0.525512,0.244464,0.351157,0.356406,...,0.692310,0.771861,0.286356,0.480954,0.525512,0.244464,0.877158,0.873199,0.072991,0.502461


In [7]:
train_time_rows = []

if not RUN_TRAIN_ABLATIONS:
    print('Skipped train-time loss/augmentation ablations. Set RUN_TRAIN_ABLATIONS=1 to execute them.')
else:
    split_hash = SPLIT_HASH_TXT.read_text().strip() if SPLIT_HASH_TXT.exists() else None
    train_out_root = OUT_DIR / 'train_time'
    train_out_root.mkdir(parents=True, exist_ok=True)

    variants = [
        {
            'name': 'baseline_bce_dice',
            'loss_name': 'bce_dice',
            'hflip_prob': 0.5,
            'vflip_prob': 0.2,
            'color_jitter_prob': 0.0,
        },
        {
            'name': 'loss_dice_only',
            'loss_name': 'dice',
            'hflip_prob': 0.5,
            'vflip_prob': 0.2,
            'color_jitter_prob': 0.0,
        },
        {
            'name': 'loss_focal_dice',
            'loss_name': 'focal_dice',
            'hflip_prob': 0.5,
            'vflip_prob': 0.2,
            'color_jitter_prob': 0.0,
        },
        {
            'name': 'aug_no_flip',
            'loss_name': 'bce_dice',
            'hflip_prob': 0.0,
            'vflip_prob': 0.0,
            'color_jitter_prob': 0.0,
        },
        {
            'name': 'aug_flip_colorjitter',
            'loss_name': 'bce_dice',
            'hflip_prob': 0.5,
            'vflip_prob': 0.2,
            'color_jitter_prob': 0.4,
        },
    ]

    for var in variants:
        run_name = var['name']
        run_out = train_out_root / run_name
        print(f'\nTrain-time ablation: {run_name}')

        cfg = TrainConfig(
            seed=42,
            image_size=TRAIN_ABL_IMAGE_SIZE,
            batch_size=TRAIN_ABL_BATCH_SIZE,
            num_workers=TRAIN_ABL_NUM_WORKERS,
            epochs=TRAIN_ABL_EPOCHS,
            lr=TRAIN_ABL_LR,
            weight_decay=TRAIN_ABL_WEIGHT_DECAY,
            threshold=0.5,
            compute_hd95=False,
            save_pred_masks=False,
            loss_name=var['loss_name'],
            hflip_prob=var['hflip_prob'],
            vflip_prob=var['vflip_prob'],
            color_jitter_prob=var['color_jitter_prob'],
            color_jitter_strength=0.2,
        )

        if TRAIN_ABL_MODEL == 'deeplabv3_resnet50':
            model = build_deeplab_resnet50(num_classes=1)
            model_name = 'deeplabv3_resnet50'
        else:
            model = UNetSmall(in_ch=3, out_ch=1, base=32)
            model_name = 'unet_small'

        try:
            results = train_and_evaluate(
                model=model,
                model_name=model_name,
                root=ROOT,
                out_dir=run_out,
                cfg=cfg,
                metadata_df=meta_df,
                split_hash=split_hash,
                max_train=TRAIN_MAX_TRAIN,
                max_val=TRAIN_MAX_VAL,
                max_test=TRAIN_MAX_TEST,
            )

            test_summary = results.get('test', {}) if isinstance(results, dict) else {}
            row = {
                'run_name': run_name,
                'model_name': model_name,
                'loss_name': cfg.loss_name,
                'hflip_prob': cfg.hflip_prob,
                'vflip_prob': cfg.vflip_prob,
                'color_jitter_prob': cfg.color_jitter_prob,
                'epochs': cfg.epochs,
                'image_size': cfg.image_size,
            }
            row.update({f'test_{k}': v for k, v in test_summary.items()})
            train_time_rows.append(row)
            print('  done; test_dice_mean=', test_summary.get('dice_mean'))
        except Exception as e:
            train_time_rows.append({'run_name': run_name, 'error': str(e)})
            print('  FAILED:', e)

train_time_df = pd.DataFrame(train_time_rows)
train_time_csv = OUT_DIR / 'train_time_ablation_summary.csv'
train_time_df.to_csv(train_time_csv, index=False)
print('Saved:', train_time_csv)
display(train_time_df)


Skipped train-time loss/augmentation ablations. Set RUN_TRAIN_ABLATIONS=1 to execute them.
Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/train_time_ablation_summary.csv


""


In [8]:
status = {
    'threshold_sweep_csv': str(sweep_csv),
    'threshold_best_csv': str(best_csv),
    'train_time_ablation_csv': str(train_time_csv),
    'n_threshold_rows': int(len(sweep_df)),
    'n_threshold_failures': int(len(fail_df)),
    'run_train_ablations': bool(RUN_TRAIN_ABLATIONS),
}
status_path = OUT_DIR / 'status.json'
status_path.write_text(json.dumps(status, indent=2))
print('Saved:', status_path)
print(json.dumps(status, indent=2))


Saved: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/status.json
{
  "threshold_sweep_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/threshold_sweep.csv",
  "threshold_best_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/threshold_best_per_run.csv",
  "train_time_ablation_csv": "/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/3_generalization_and_ablation/out/ablations/train_time_ablation_summary.csv",
  "n_threshold_rows": 15,
  "n_threshold_failures": 0,
  "run_train_ablations": false
}
